In [0]:
from pyspark.sql import functions as F

# ------------------------------------------------------------------
# 1) LER O BRONZE E FILTRAR
# ------------------------------------------------------------------
bronze = spark.table("bronze_mortalidade_sim")
print("Total de linhas no bronze:", bronze.count())

# Filtro de segurança (hoje não remove nada, mas protege se a fonte mudar)
silver = bronze.filter(F.col("TIPOBITO") == "2")
print("Linhas depois do filtro TIPOBITO=2:", silver.count())

# ------------------------------------------------------------------
# 2) DECODIFICAR IDADE -> IDADE_ANOS E FAIXA_ETARIA
# ------------------------------------------------------------------
idade_int = F.col("IDADE").cast("int")
unidade = (idade_int / 100).cast("int")
valor = idade_int % 100

silver = silver.withColumn(
    "IDADE_ANOS",
    F.when(unidade == 4, valor)
     .when(unidade == 5, 100 + valor)
     .when(unidade.isin(0, 1, 2, 3), 0)
     .otherwise(None)
)

silver = silver.withColumn(
    "FAIXA_ETARIA",
    F.when(F.col("IDADE_ANOS").isNull(), "Ignorada")
     .when(F.col("IDADE_ANOS") <= 14, "0-14")
     .when(F.col("IDADE_ANOS") <= 29, "15-29")
     .when(F.col("IDADE_ANOS") <= 44, "30-44")
     .when(F.col("IDADE_ANOS") <= 59, "45-59")
     .when(F.col("IDADE_ANOS") <= 74, "60-74")
     .otherwise("75+")
)

# ------------------------------------------------------------------
# 3) CLASSIFICAR TEMA (CIRCOBITO como fonte primária, CID-10 como fallback)
# ------------------------------------------------------------------
causa = F.col("CAUSABAS")
letra = F.substring(causa, 1, 1)
num = F.substring(causa, 2, 2).cast("int")
circ = F.col("CIRCOBITO").cast("int")

is_externa = letra.isin("V", "W", "X", "Y")

circ_conhecido = F.coalesce(circ.isin(1, 2, 3, 4), F.lit(False))
fallback = is_externa & (~circ_conhecido)

silver = silver.withColumn(
    "TEMA",
    F.when(causa.isNull(), "Não classificado")
     .when(~is_externa, "Saúde")
     .when(circ == 1, "Acidente")
     .when(circ == 2, "Suicídio")
     .when(circ == 3, "Homicídio")
     .when(circ == 4, "Outra causa externa")
     .when(fallback & (letra == "X") & num.between(60, 84), "Suicídio")
     .when(fallback & (((letra == "X") & num.between(85, 99)) | ((letra == "Y") & num.between(0, 9))), "Homicídio")
     .when(fallback & ((letra == "V") | (letra == "W") | ((letra == "X") & (num <= 59))), "Acidente")
     .otherwise("Não classificado")
)

# ------------------------------------------------------------------
# 4) SUBCLASSIFICAR OS TEMAS POR CAPÍTULO/MECANISMO DO CID-10
#    (Saúde: por capítulo do CID-10 | Acidente/Suicídio/Homicídio:
#    por mecanismo, usando as sub-faixas V/W/X/Y)
# ------------------------------------------------------------------
silver = silver.withColumn(
    "SUBTEMA",

    # ---- SAÚDE: por capítulo do CID-10 (igual já tínhamos) ----
    F.when((F.col("TEMA") == "Saúde") & causa.isNull(), "Não classificado")
     .when((F.col("TEMA") == "Saúde") & (letra == "U") & (num == 7), "COVID-19")
     .when((F.col("TEMA") == "Saúde") & letra.isin("A", "B"), "Doenças infecciosas e parasitárias")
     .when((F.col("TEMA") == "Saúde") & ((letra == "C") | ((letra == "D") & (num <= 48))), "Neoplasias (câncer)")
     .when((F.col("TEMA") == "Saúde") & (letra == "E"), "Doenças endócrinas, nutricionais e metabólicas")
     .when((F.col("TEMA") == "Saúde") & (letra == "F"), "Transtornos mentais e comportamentais")
     .when((F.col("TEMA") == "Saúde") & (letra == "G"), "Doenças do sistema nervoso")
     .when((F.col("TEMA") == "Saúde") & (letra == "I"), "Doenças do aparelho circulatório")
     .when((F.col("TEMA") == "Saúde") & (letra == "J"), "Doenças do aparelho respiratório")
     .when((F.col("TEMA") == "Saúde") & (letra == "K"), "Doenças do aparelho digestivo")
     .when((F.col("TEMA") == "Saúde") & letra.isin("P", "Q"), "Causas perinatais e malformações congênitas")
     .when((F.col("TEMA") == "Saúde") & (letra == "R"), "Causas mal definidas")
     .when(F.col("TEMA") == "Saúde", "Outras doenças")

     # ---- ACIDENTE: por mecanismo (V01-X59) ----
     .when((F.col("TEMA") == "Acidente") & (letra == "V"), "Acidente de trânsito/transporte")
     .when((F.col("TEMA") == "Acidente") & (letra == "W") & num.between(0, 19), "Queda")
     .when((F.col("TEMA") == "Acidente") & (letra == "W") & num.between(65, 74), "Afogamento")
     .when((F.col("TEMA") == "Acidente") & (letra == "W") & num.between(75, 84), "Sufocação/engasgo")
     .when((F.col("TEMA") == "Acidente") & (letra == "X") & num.between(0, 9), "Queimadura/fogo")
     .when((F.col("TEMA") == "Acidente") & (letra == "X") & num.between(40, 49), "Envenenamento acidental")
     .when(F.col("TEMA") == "Acidente", "Outros acidentes")

     # ---- SUICÍDIO: por mecanismo (X60-X84) ----
     .when((F.col("TEMA") == "Suicídio") & (letra == "X") & num.between(60, 69), "Autointoxicação/envenenamento")
     .when((F.col("TEMA") == "Suicídio") & (letra == "X") & (num == 70), "Enforcamento")
     .when((F.col("TEMA") == "Suicídio") & (letra == "X") & num.between(72, 74), "Arma de fogo")
     .when((F.col("TEMA") == "Suicídio") & (letra == "X") & (num == 78), "Objeto cortante")
     .when((F.col("TEMA") == "Suicídio") & (letra == "X") & (num == 80), "Salto de lugar elevado")
     .when(F.col("TEMA") == "Suicídio", "Outros meios")

     # ---- HOMICÍDIO: por mecanismo (X85-Y09) ----
     .when((F.col("TEMA") == "Homicídio") & (letra == "X") & num.between(93, 95), "Arma de fogo")
     .when((F.col("TEMA") == "Homicídio") & (letra == "X") & (num == 99), "Objeto cortante/perfurante")
     .when((F.col("TEMA") == "Homicídio") & (letra == "X") & (num == 91), "Enforcamento/estrangulamento")
     .when((F.col("TEMA") == "Homicídio") & (letra == "Y") & (num == 3), "Força corporal/agressão física")
     .when(F.col("TEMA") == "Homicídio", "Outros meios")

     # ---- Outra causa externa / Não classificado: sem subtema (igual antes) ----
     .otherwise(None)
)

# ------------------------------------------------------------------
# 5) RÓTULOS DE RACACOR E SEXO
# ------------------------------------------------------------------
silver = silver.withColumn(
    "RACACOR_DESC",
    F.when(F.col("RACACOR") == "1", "Branca")
     .when(F.col("RACACOR") == "2", "Preta")
     .when(F.col("RACACOR") == "3", "Amarela")
     .when(F.col("RACACOR") == "4", "Parda")
     .when(F.col("RACACOR") == "5", "Indígena")
     .otherwise("Ignorada")
)

silver = silver.withColumn(
    "SEXO_DESC",
    F.when(F.col("SEXO") == "1", "Masculino")
     .when(F.col("SEXO") == "2", "Feminino")
     .otherwise("Ignorado")
)

# ------------------------------------------------------------------
# 6) VALIDAÇÃO: TEMA (mesmos números de sempre, não deve ter mudado)
# ------------------------------------------------------------------
display(
    silver.filter(F.col("ANO_REFERENCIA") == "2023")
          .groupBy("TEMA")
          .count()
          .orderBy(F.desc("count"))
)

# ------------------------------------------------------------------
# 6b) VALIDAÇÃO NOVA: distribuição dos subtemas recém-criados
# ------------------------------------------------------------------
display(
    silver.filter(F.col("ANO_REFERENCIA") == "2023")
          .filter(F.col("TEMA").isin("Acidente", "Suicídio", "Homicídio"))
          .groupBy("TEMA", "SUBTEMA")
          .count()
          .orderBy("TEMA", F.desc("count"))
)

# ------------------------------------------------------------------
# 7) GRAVAR A TABELA SILVER
# ------------------------------------------------------------------
(silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_mortalidade_sim"))
print("Tabela silver gravada:", silver.count(), "linhas")